# Final assembly: output/index.html

Stitches together the narrative from `30-narrative.ipynb` with the four HTML fragments
produced by `31-render.ipynb` into a single-page story at `../output/index.html`.

Each fragment (`fig1.frag.html` – `fig4.frag.html`) is embedded inline — no iframes.
Leaflet CSS/JS is included once in `<head>` (blocking, so inline fig scripts can use `L`); asset paths
(`assets/blocks.pbf`, `placement_grids/`) resolve relative to `output/`.

In [13]:
import json
import re
from pathlib import Path

NARRATIVE = Path("../notebooks/30-narrative.ipynb")
OUTPUT    = Path("../output")

# ── Load narrative markdown cells ─────────────────────────────────────────────
nb = json.loads(NARRATIVE.read_text())
all_md = [
    "".join(c["source"])
    for c in nb["cells"]
    if c["cell_type"] == "markdown"
]
prose = [c for c in all_md if c.strip() and not c.strip().startswith("**FIG")]
# prose[0] = intro (begins with "# The care gap…")
# prose[1] = "## Bigger than Hardwick" / "## Getting there from here"
# prose[2] = "## The weight of the problem"
# prose[3] = "## The case for Hardwick"
# prose[4] = "## Where else is at risk?"
# prose[5] = "## Methods and limitations"

# ── Extract captions from FIG placeholder cells ───────────────────────────────
fig_captions: dict[int, str] = {}
for cell in all_md:
    m = re.match(r'\*\*FIG (\d+)\*\*', cell.strip())
    if not m:
        continue
    fig_num = int(m.group(1))
    cap_match = re.search(r'^Caption:\s*(.+)$', cell, re.MULTILINE)
    fig_captions[fig_num] = cap_match.group(1).strip() if cap_match else ""


def figcaption(fig_num: int) -> str:
    cap = fig_captions.get(fig_num, "")
    return f'<figcaption>{cap}</figcaption>\n' if cap else ""

# ── Simple markdown → HTML (headings, paragraphs, bold, links) ───────────────
def md_to_html(text: str) -> str:
    text = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', text)
    text = re.sub(
        r'\[([^\]]+)\]\(([^)(]*(?:\([^)(]*\)[^)(]*)*)\)',
        r'<a href="\2" target="_blank" rel="noopener">\1</a>',
        text,
    )
    parts = []
    for para in re.split(r'\n{2,}', text.strip()):
        para = para.strip()
        if not para:
            continue
        m = re.match(r'^(#{1,6})\s+(.+)$', para, re.DOTALL)
        if m:
            lvl = len(m.group(1))
            parts.append(f'<h{lvl}>{m.group(2).strip()}</h{lvl}>')
        else:
            parts.append(f'<p>{para.replace(chr(10), " ")}</p>')
    return "\n".join(parts)

# ── Extract page title; convert intro body (strip the h1 title line) ──────────
title_match = re.match(r'^#\s+(.+)', prose[0])
page_title  = title_match.group(1).strip() if title_match else "The care gap in the Kingdom"
intro_body  = re.sub(r'^#\s+.+\n?', '', prose[0], count=1).strip()
intro_html  = md_to_html(intro_body)

# ── Load fragments ─────────────────────────────────────────────────────────────
frags = {i: (OUTPUT / f"fig{i}.frag.html").read_text() for i in range(1, 5)}

# ── Compose page using a template (avoids f-string brace escaping in CSS) ─────
LEAFLET_CSS = "https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"
LEAFLET_JS  = "https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"

CSS = """
*, *::before, *::after {box-sizing:border-box}
body {
  margin: 0;
  font-family: Georgia, 'Times New Roman', serif;
  background: #fafaf8;
  color: #222;
  line-height: 1.75;
}
.prose {
  max-width: 720px;
  margin: 0 auto;
  padding: 0 1.25rem;
}
header {
  max-width: 720px;
  margin: 3rem auto 2rem;
  padding: 0 1.25rem;
}
header h1 {
  font-size: 2rem;
  font-weight: 700;
  margin: 0 0 .4rem;
  line-height: 1.2;
}
.byline {
  font-size: .85rem;
  color: #888;
  font-family: system-ui, sans-serif;
}
h2 {
  font-size: 1.25rem;
  font-weight: 700;
  margin: 2.5rem 0 .75rem;
  color: #1a3a5c;
}
p { margin: 0 0 1rem; }
a { color: #1a5276; }
.fig-full {
  width: calc(100vw - 4rem);
  position: relative;
  left: 50%;
  transform: translateX(-50%);
  margin: 2rem 0;
}
figure.fig-full figcaption {
  max-width: 720px;
  margin: .5rem auto 0;
  padding: 0 1.25rem;
  font-size: .8rem;
  color: #666;
  font-family: system-ui, sans-serif;
  font-style: italic;
}
hr.fin {
  border: none;
  border-top: 1px solid #ddd;
  max-width: 720px;
  margin: 3rem auto;
}
"""

page = (
    '<!DOCTYPE html>\n<html lang="en">\n<head>\n'
    '<meta charset="utf-8">\n'
    '<meta name="viewport" content="width=device-width,initial-scale=1">\n'
    f'<title>{page_title}</title>\n'
    f'<link rel="stylesheet" href="{LEAFLET_CSS}">\n'
    f'<script src="{LEAFLET_JS}"></script>\n'
    f'<style>{CSS}</style>\n'
    '</head>\n<body>\n'
    f'<header>\n  <h1>{page_title}</h1>\n'
    '  <div class="byline">Joe Nudell &nbsp;&middot;&nbsp; March 2026</div>\n'
    '</header>\n<main>\n\n'
    f'<div class="prose">\n{intro_html}\n</div>\n\n'
    '<figure class="fig-full">\n'
    + frags[1] + '\n'
    + figcaption(1)
    + '</figure>\n\n'
    f'<div class="prose">\n{md_to_html(prose[1])}\n</div>\n\n'
    f'<div class="prose">\n{md_to_html(prose[2])}\n</div>\n\n'
    '<figure class="fig-full">\n'
    + frags[2] + '\n'
    + figcaption(2)
    + '</figure>\n\n'
    f'<div class="prose">\n{md_to_html(prose[3])}\n</div>\n\n'
    '<figure class="fig-full">\n'
    + frags[3] + '\n'
    + figcaption(3)
    + '</figure>\n\n'
    '<figure class="fig-full">\n'
    + frags[4] + '\n'
    + figcaption(4)
    + '</figure>\n\n'
    f'<div class="prose">\n{md_to_html(prose[4])}\n</div>\n\n'
    f'<div class="prose">\n{md_to_html(prose[5])}\n</div>\n\n'
    '<hr class="fin">\n'
    '</main>\n'
    '</body>\n</html>\n'
)

out = OUTPUT / "index.html"
out.write_text(page, encoding="utf-8")
size_kb = out.stat().st_size / 1024
print(f"Wrote {out}  ({size_kb:.0f} KB)")

for frag_id in ["fig1", "fig2", "fig3", "fig4"]:
    assert f'id="{frag_id}"' in page, f"Missing fragment #{frag_id}"
print("All four fragments present ✓")

Wrote ../output/index.html  (449 KB)
All four fragments present ✓
